<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning-/blob/main/Day3/KernelPCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Kernal PCA on ev_Charging_patterns



In [ ]:
import pandas as pd
df = pd.read_csv('/content/ev_charging_patterns.csv')

In [ ]:
df.columns

Index(['User ID', 'Vehicle Model', 'Battery Capacity (kWh)',
       'Charging Station ID', 'Charging Station Location',
       'Charging Start Time', 'Charging End Time', 'Energy Consumed (kWh)',
       'Charging Duration (hours)', 'Charging Rate (kW)',
       'Charging Cost (USD)', 'Time of Day', 'Day of Week',
       'State of Charge (Start %)', 'State of Charge (End %)',
       'Distance Driven (since last charge) (km)', 'Temperature (°C)',
       'Vehicle Age (years)', 'Charger Type', 'User Type'],
      dtype='object')

In [ ]:
df.isna().sum()

,0
User ID,0
Vehicle Model,0
Battery Capacity (kWh),0
Charging Station ID,0
Charging Station Location,0
Charging Start Time,0
Charging End Time,0
Energy Consumed (kWh),66
Charging Duration (hours),0
Charging Rate (kW),66


In [ ]:
df.dtypes

,0
User ID,object
Vehicle Model,object
Battery Capacity (kWh),float64
Charging Station ID,object
Charging Station Location,object
Charging Start Time,object
Charging End Time,object
Energy Consumed (kWh),float64
Charging Duration (hours),float64
Charging Rate (kW),float64


In [ ]:
df.nunique()

,0
User ID,1320
Vehicle Model,5
Battery Capacity (kWh),147
Charging Station ID,462
Charging Station Location,5
Charging Start Time,1320
Charging End Time,1309
Energy Consumed (kWh),1254
Charging Duration (hours),1320
Charging Rate (kW),1254


In [ ]:
df.drop(columns=['User ID','Charging Station ID'],inplace=True)

In [ ]:
df.shape

(1320, 18)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(1320, 18)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

(1131, 18)

In [ ]:
df.drop(columns=['Charging Start Time','Charging End Time'],inplace=True)

In [ ]:
df.shape

(1131, 16)

In [ ]:
df_ohe = pd.get_dummies(df)

In [ ]:
df_ohe.shape

(1131, 37)

In [ ]:
X = df_ohe

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test = train_test_split(X,test_size=0.3,random_state=7)
X_train.shape,X_test.shape

((791, 37), (340, 37))

## Standard Scaling by formula

In [25]:
X_train = (X_train-X_train.mean())/ X_train.std()
X_test = (X_test-X_train.mean())/ X_train.std()

In [26]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel="poly",degree=2,random_state=7)
kpca.fit(X_train)

KernelPCA(degree=2, kernel='poly', random_state=7)

In [27]:
kpca.eigenvalues_.shape

(474,)

In [30]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel="rbf",random_state=7)
kpca.fit(X_train)

KernelPCA(kernel='rbf', random_state=7)

In [31]:
kpca.eigenvalues_.shape

(790,)

In [34]:
def outlier_impute_iqr(df,col):
  df1 = df.copy()
  print("Column --> ",col)
  q1,q3 = df1[col].quantile([0.25,0.75])
  iqr = q3-q1
  LB = q1 - 1.5*iqr
  UB = q3 + 1.5*iqr
  df1.loc[(df1[col]<UB),col] = UB
  df1.loc[(df1[col]>LB),col] = LB
  print(LB, UB, df1[col].min(), df1[col].max())
  return df1

In [35]:
cont_col = X_train.columns[X_train.nunique() > 15]
cont_col

Index(['Battery Capacity (kWh)', 'Energy Consumed (kWh)',
       'Charging Duration (hours)', 'Charging Rate (kW)',
       'Charging Cost (USD)', 'State of Charge (Start %)',
       'State of Charge (End %)', 'Distance Driven (since last charge) (km)',
       'Temperature (°C)', 'Vehicle Age (years)'],
      dtype='object')

In [37]:
X_train_im_io = X_train.copy()
for col in cont_col:
  X_train_im_io = outlier_impute_iqr(X_train_im_io,col)

Column -->  Battery Capacity (kWh)
-2.324169622723589 2.2284954219957336 -2.324169622723589 -2.324169622723589
Column -->  Energy Consumed (kWh)
-3.3271059378496624 3.3446660603796357 -3.3271059378496624 -3.3271059378496624
Column -->  Charging Duration (hours)
-3.2682970989850366 3.2263772904346073 -3.2682970989850366 -3.2682970989850366
Column -->  Charging Rate (kW)
-3.3090438368229513 3.2641130018008724 -3.3090438368229513 -3.3090438368229513
Column -->  Charging Cost (USD)
-3.421879164353652 3.433313584157328 -3.421879164353652 -3.421879164353652
Column -->  State of Charge (Start %)
-3.4419401990972815 3.399515140636539 -3.4419401990972815 -3.4419401990972815
Column -->  State of Charge (End %)
-3.1149311902721983 3.1267312251564086 -3.1149311902721983 -3.1149311902721983
Column -->  Distance Driven (since last charge) (km)
-3.4765358075685255 3.505793079749683 -3.4765358075685255 -3.4765358075685255
Column -->  Temperature (°C)
-3.304466717677972 3.325483864301752 -3.30446671767